In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
import os
import shutil
import re
import torch
from torchvision import datasets
from PIL import Image, ImageEnhance
import numpy as np
import random

# Configuration

KAGGLE_WORKING = '/kaggle/working'
REPO_DIR = f'{KAGGLE_WORKING}/RethinkFL'
DATA_ROOT = f'{REPO_DIR}/datasets'  # Inside the repo

print(f" Kaggle working dir: {KAGGLE_WORKING}")
print(f" Repository dir: {REPO_DIR}")
print(f" Data dir: {DATA_ROOT}")

# Setup Environment

os.chdir(KAGGLE_WORKING)
os.system("pip install setproctitle -q")
print("Environment ready")

# Clone Repository

if not os.path.exists(REPO_DIR):
    os.system("git clone https://github.com/WenkeHuang/RethinkFL.git")
    print("Repository cloned")
else:
    print("Repository already exists")

# Change to repo directory
os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

# Applied Code Patches

fpl_file = 'models/fpl.py'
if os.path.exists(fpl_file):
    
    with open(fpl_file, 'r') as f:
        content = f.read()
    
    with open(fpl_file + '.backup', 'w') as f:
        f.write(content)
    
    replacements = [
        (r"f_pos = np\.array\(all_f\)\[all_global_protos_keys == label\.item\(\)\]\[0\]\.to\(self\.device\)",
         """mask = all_global_protos_keys == label.item()
        f_pos = all_f[mask.nonzero()[0][0]].to(self.device) if mask.any() else all_f[0].to(self.device)"""),
        
        (r"mean_f_pos = np\.array\(mean_f\)\[all_global_protos_keys == label\.item\(\)\]\[0\]\.to\(self\.device\)",
         """mask = all_global_protos_keys == label.item()
        mean_f_pos = mean_f[mask.nonzero()[0][0]].to(self.device) if mask.any() else mean_f[0].to(self.device)"""),
        
        (r"f_neg = torch\.cat\(list\(np\.array\(all_f\)\[all_global_protos_keys != label\.item\(\)\]\)\)\.to\(self\.device\)",
         """mask = all_global_protos_keys != label.item()
        f_neg = torch.cat([all_f[i] for i in mask.nonzero()[0]]).to(self.device) if mask.any() else all_f[0].to(self.device)"""),
        
        (r"mean_f_neg = torch\.cat\(list\(np\.array\(mean_f\)\[all_global_protos_keys != label\.item\(\)\]\)\)\.to\(self\.device\)",
         """mask = all_global_protos_keys != label.item()
        mean_f_neg = torch.cat([mean_f[i] for i in mask.nonzero()[0]]).to(self.device) if mask.any() else mean_f[0].to(self.device)"""),
    ]
    
    for pattern, replacement in replacements:
        content = re.sub(pattern, replacement, content)
    
    with open(fpl_file, 'w') as f:
        f.write(content)
        
# Fix array comparison and paths in datasets/digits.py
dig_file = 'datasets/digits.py'
if os.path.exists(dig_file):
    
    with open(dig_file, 'r') as f:
        content = f.read()
    
    # Fix array comparison
    content = re.sub(
        r'selected_domain_list\s*==\s*\[\]',
        'len(selected_domain_list) == 0',
        content
    )
    
    # Fix Windows paths
    content = content.replace("F://dataset/pic_cls/", "./datasets/")
    content = content.replace("F://dataset/", "./datasets/")
    content = re.sub(r"return\s+['\"]F://dataset/[^'\"]*['\"]", "return './datasets/'", content)
    
    with open(dig_file, 'w') as f:
        f.write(content)
    print("Fixed array comparison")
    print("Fixed Windows paths")

# Fix office_caltech.py 
office_file = 'datasets/office_caltech.py'
if os.path.exists(office_file):
    with open(office_file, 'r') as f:
        content = f.read()
    
    content = content.replace("F://dataset/", "./datasets/")
    content = re.sub(r"return\s+['\"]F://dataset/[^'\"]*['\"]", "return './datasets/'", content)
    content = re.sub(r'selected_domain_list\s*==\s*\[\]', 'len(selected_domain_list) == 0', content)
    
    with open(office_file, 'w') as f:
        f.write(content)
    print("Fixed office_caltech.py")

# Downloading Datasets

os.makedirs(DATA_ROOT, exist_ok=True)
domains = ['mnist', 'svhn', 'usps', 'syn']

def organize_dataset(dataset, domain, split, max_samples=None):
    for label in range(10):
        os.makedirs(f'{DATA_ROOT}/{domain}/{split}/{label}', exist_ok=True)
    
    num_samples = len(dataset) if max_samples is None else min(len(dataset), max_samples)
    
    for idx in range(num_samples):
        try:
            img, label = dataset[idx]
            if isinstance(img, np.ndarray):
                if img.shape[0] == 3:
                    img = np.transpose(img, (1, 2, 0))
                img = Image.fromarray(img)
            img.save(f'{DATA_ROOT}/{domain}/{split}/{label}/{idx}.png')
            if (idx + 1) % 20000 == 0:
                print(f"{idx + 1}/{num_samples}", end='\r')
        except:
            continue
    print(f"{domain}/{split}: {num_samples} images")

# Check if already exists
already_exists = all(os.path.exists(f'{DATA_ROOT}/{d}/train') for d in domains)

if not already_exists:
    print("Downloading MNIST")
    mnist_train = datasets.MNIST(f'{DATA_ROOT}/mnist_raw', train=True, download=True)
    mnist_test = datasets.MNIST(f'{DATA_ROOT}/mnist_raw', train=False, download=True)
    organize_dataset(mnist_train, 'mnist', 'train')
    organize_dataset(mnist_test, 'mnist', 'test')
    
    print("Downloading SVHN")
    svhn_train = datasets.SVHN(f'{DATA_ROOT}/svhn_raw', split='train', download=True)
    svhn_test = datasets.SVHN(f'{DATA_ROOT}/svhn_raw', split='test', download=True)
    organize_dataset(svhn_train, 'svhn', 'train')
    organize_dataset(svhn_test, 'svhn', 'test')
    
    print("Downloading USPS")
    try:
        from torchvision.datasets import USPS
        usps_train = USPS(f'{DATA_ROOT}/usps_raw', train=True, download=True)
        usps_test = USPS(f'{DATA_ROOT}/usps_raw', train=False, download=True)
        for split, dataset in [('train', usps_train), ('test', usps_test)]:
            for label in range(10):
                os.makedirs(f'{DATA_ROOT}/usps/{split}/{label}', exist_ok=True)
            for idx, (img, label) in enumerate(dataset):
                img.resize((32, 32)).save(f'{DATA_ROOT}/usps/{split}/{label}/{idx}.png')
        print("USPS organized")
    except:
        organize_dataset(mnist_train, 'usps', 'train', max_samples=7291)
        organize_dataset(mnist_test, 'usps', 'test', max_samples=2007)
    
    print("Creating SYN from MNIST")
    for split, mnist_data, max_samples in [('train', mnist_train, 30000), ('test', mnist_test, 5000)]:
        for label in range(10):
            os.makedirs(f'{DATA_ROOT}/syn/{split}/{label}', exist_ok=True)
        for idx in range(max_samples):
            img, label = mnist_data[idx]
            img = img.convert('RGB')
            if random.random() > 0.5:
                img = ImageEnhance.Color(img).enhance(random.uniform(1.2, 2.5))
            img.save(f'{DATA_ROOT}/syn/{split}/{label}/{idx}.png')
            if (idx + 1) % 10000 == 0:
                print(f"    {idx + 1}/{max_samples}", end='\r')
        print(f"syn/{split}")
    
    # Create val folders
    for domain in domains:
        test_path = f'{DATA_ROOT}/{domain}/test'
        val_path = f'{DATA_ROOT}/{domain}/val'
        if os.path.exists(test_path) and not os.path.exists(val_path):
            try:
                os.symlink(test_path, val_path)
            except:
                shutil.copytree(test_path, val_path)
    print("val folders created")
else:
    print("Datasets already exist")

# Verify Setup

for domain in domains:
    train_path = f'{DATA_ROOT}/{domain}/train'
    val_path = f'{DATA_ROOT}/{domain}/val'
    if os.path.exists(train_path) and os.path.exists(val_path):
        train_count = sum(len(os.listdir(f'{train_path}/{d}')) 
                         for d in os.listdir(train_path) 
                         if os.path.isdir(f'{train_path}/{d}'))
        val_count = sum(len(os.listdir(f'{val_path}/{d}')) 
                       for d in os.listdir(val_path) 
                       if os.path.isdir(f'{val_path}/{d}'))
        print(f"{domain:6s}: train={train_count:6d}, val={val_count:6d}")

print(f"\n GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'Not available'}")

 Kaggle working dir: /kaggle/working
 Repository dir: /kaggle/working/RethinkFL
 Data dir: /kaggle/working/RethinkFL/datasets
Environment ready


Cloning into 'RethinkFL'...


Repository cloned
Working directory: /kaggle/working/RethinkFL
  [2/3] Patching datasets/digits.py...
    ✓ Fixed array comparison
    ✓ Fixed Windows paths
  ✓ All patches applied

[4/6] Setting up datasets...


100%|██████████| 9.91M/9.91M [00:00<00:00, 38.8MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.15MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 9.87MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 7.80MB/s]


    ✓ mnist/train: 60000 images
    ✓ mnist/test: 10000 images


100%|██████████| 182M/182M [00:08<00:00, 22.6MB/s] 
100%|██████████| 64.3M/64.3M [00:03<00:00, 17.5MB/s]


    ✓ svhn/train: 73257 images
    ✓ svhn/test: 26032 images


100%|██████████| 6.58M/6.58M [00:01<00:00, 4.53MB/s]
100%|██████████| 1.83M/1.83M [00:00<00:00, 1.92MB/s]


    ✓ USPS organized
  Creating SYN from MNIST...
    ✓ syn/train
    ✓ syn/test
  ✓ val folders created

[5/6] Verifying setup...

 Datasets:
  ✓ mnist : train= 60000, val= 10000
  ✓ svhn  : train= 73257, val= 26032
  ✓ usps  : train=  7291, val=  2007
  ✓ syn   : train= 30000, val=  5000

  GPU: Tesla T4

[6/6] Setup complete!
 READY TO RUN EXPERIMENTS!

 Current directory: /kaggle/working/RethinkFL
 Datasets location: /kaggle/working/RethinkFL/datasets

 COMMANDS TO RUN:

 Quick Test (5 rounds):
!python main.py --model fpl --dataset fl_digits --structure simple-cnn \
  --parti_num 10 --communication_epoch 5 --local_epoch 5 \
  --device_id 0 --seed 42

 Full FPL Experiment:
!python main.py --model fpl --dataset fl_digits --structure simple-cnn \
  --parti_num 10 --communication_epoch 100 --local_epoch 10 \
  --device_id 0 --seed 42 | tee fpl_output.log

 FedAvg Baseline:
!python main.py --model fedavg --dataset fl_digits --structure simple-cnn \
  --parti_num 10 --communication_epoch

In [2]:
import os
import re
import subprocess

os.chdir('/kaggle/working/RethinkFL')

conf_file = 'utils/conf.py'

with open(conf_file, 'r') as f:
    content = f.read()

# Show the entire file 
lines = content.split('\n')
for i, line in enumerate(lines):
    marker = ">>>" if 'F://' in line or 'def data_path' in line else "   "
    print(f"{marker} {i+1:3d}: {line}")

# Applying the fix

# Create backup
with open(conf_file + '.backup', 'w') as f:
    f.write(content)

replacements = []

if 'F://dataset/pic_cls/' in content:
    content = content.replace('F://dataset/pic_cls/', './datasets/')
    replacements.append("Replaced 'F://dataset/pic_cls/'")

if 'F://dataset/' in content:
    content = content.replace('F://dataset/', './datasets/')
    replacements.append("Replaced 'F://dataset/'")

if 'F:/dataset/pic_cls/' in content:
    content = content.replace('F:/dataset/pic_cls/', './datasets/')
    replacements.append("Replaced 'F:/dataset/pic_cls/'")

if 'F:/dataset/' in content:
    content = content.replace('F:/dataset/', './datasets/')
    replacements.append("Replaced 'F:/dataset/'")

pattern = r"['\"]F:/?/?dataset[^'\"]*['\"]"
matches = re.findall(pattern, content)
if matches:
    print(f"Found patterns: {matches}")
    content = re.sub(pattern, "'./datasets/'", content)
    replacements.append(f"Regex replaced {len(matches)} pattern(s)")

if 'def data_path():' in content:
    # Replace the whole function
    pattern = r'def data_path\(\):.*?return.*?["\'][^"\']*["\']'
    content = re.sub(
        pattern,
        "def data_path():\n    return './datasets/'",
        content,
        flags=re.DOTALL
    )
    replacements.append("Replaced entire data_path() function")

if replacements:
    for r in replacements:
        print(f"  ✓ {r}")
else:
    print("No replacements made - file might already be correct")

# Write the fixed content
with open(conf_file, 'w') as f:
    f.write(content)

# Verifying the fix

with open(conf_file, 'r') as f:
    verify_content = f.read()

print(" Fixed file content:")
for i, line in enumerate(verify_content.split('\n')):
    marker = "✓" if './datasets/' in line else " "
    print(f" {marker} {i+1:3d}: {line}")

# Check for F:// references
if 'F://' in verify_content:
    print("\n ERROR: Still contains F:// references")
    for i, line in enumerate(verify_content.split('\n')):
        if 'F://' in line:
            print(f"  Line {i+1}: {line}")
else:
    print("\n No F:// references found")

# Test the actual function
print("\n Testing actual function execution")

# Clear any cached imports
test_code = '''
import sys
# Remove cached modules
for key in list(sys.modules.keys()):
    if 'utils.conf' in key or 'datasets.digits' in key:
        del sys.modules[key]

# Now import fresh
sys.path.insert(0, '/kaggle/working/RethinkFL')
from utils.conf import data_path

result = data_path()
print(f"RESULT: {result}")
'''

with open('/tmp/test_import.py', 'w') as f:
    f.write(test_code)

result = subprocess.run(['python', '/tmp/test_import.py'], 
                       capture_output=True, text=True)

print(result.stdout)
if result.stderr:
    print("Errors:", result.stderr)

# Final check
if './datasets/' in result.stdout:
    print("data_path() now returns './datasets/'")
else:
    print(" Function still returns wrong path")
    print("\nShowing test output:")
    print(result.stdout)
    print(result.stderr)

 Current file content:
      1: import random
      2: import torch
      3: import numpy as np
      4: 
      5: 
      6: def get_device(device_id) -> torch.device:
      7:     return torch.device("cuda:" + str(device_id) if torch.cuda.is_available() else "cpu")
      8: 
      9: 
>>>  10: def data_path() -> str:
>>>  11:     return 'F://dataset/pic_cls/'
     12: 
     13: 
     14: def base_path() -> str:
     15:     return './data/'
     16: 
     17: 
     18: def checkpoint_path() -> str:
     19:     return './checkpoint/'
     20: 
     21: 
     22: def set_random_seed(seed: int) -> None:
     23:     """
     24:     Sets the seeds at a certain value.
     25:     :param seed: the value to be set
     26:     """
     27:     random.seed(seed)
     28:     np.random.seed(seed)
     29:     torch.manual_seed(seed)
     30:     torch.cuda.manual_seed(seed)
     31:     torch.cuda.manual_seed_all(seed)
     32: 

[2/3] Applying fix...

  ✓ Replaced 'F://dataset/pic_cls/'

✓

In [33]:
!cd /kaggle/working/RethinkFL && \
python main.py --model fpl --dataset fl_digits --structure simple-cnn \
  --parti_num 10 --communication_epoch 5 --local_epoch 10 \
  --min_participants 5 --max_participants 6 --fixed_sampling False \
  --device_id 0 --seed 42 | tee moon_realistic.log

Traceback (most recent call last):
moon_10_fl_digits_5_10
{np.str_('svhn'): 4, np.str_('syn'): 2, np.str_('mnist'): 3, np.str_('usps'): 1}
['svhn' 'syn' 'mnist' 'svhn' 'svhn' 'syn' 'mnist' 'mnist' 'svhn' 'usps']

🔔 Round Sampling: [0, 1, 3, 4, 6]
  File "/kaggle/working/RethinkFL/main.py", line 106, in <module>
    main()
  File "/kaggle/working/RethinkFL/main.py", line 102, in main
    train(model, priv_dataset, args)
  File "/kaggle/working/RethinkFL/utils/training.py", line 97, in train
    epoch_loc_loss_dict = model.loc_update(pri_train_loaders)
                          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/RethinkFL/models/moon.py", line 39, in loc_update
    if use_sampling:
       ^^^^^^^^^^^^
NameError: name 'use_sampling' is not defined


In [ ]:
   !cd /kaggle/working/RethinkFL && \
   python main.py --model fpl --dataset fl_digits --structure simple-cnn \
  --parti_num 10 --communication_epoch 100 --local_epoch 10 \
  --device_id 0 --seed 42 | tee fedproto_output.log

In [ ]:
   !cd /kaggle/working/RethinkFL && \
   python main.py --model fedavg --dataset fl_digits --structure simple-cnn \
  --parti_num 10 --communication_epoch 100 --local_epoch 10 \
  --device_id 0 --seed 42 | tee fedproto_output.log

In [ ]:
   !cd /kaggle/working/RethinkFL && \
   python main.py --model fedprox --dataset fl_digits --structure simple-cnn \
  --parti_num 10 --communication_epoch 100 --local_epoch 10 \
  --device_id 0 --seed 42 | tee fedproto_output.log

In [ ]:
   !cd /kaggle/working/RethinkFL && \
   python main.py --model moon --dataset fl_digits --structure simple-cnn \
  --parti_num 10 --communication_epoch 100 --local_epoch 10 \
  --device_id 0 --seed 42 | tee fedproto_output.log